In [1]:
!pip install gradio pandas requests

In [2]:
import json
import pandas as pd
import requests
import gradio as gr

# ========== DADOS MOCKADOS (Mariana) ==========
perfil = {
    "nome": "Mariana Costa",
    "idade": 29,
    "perfil_investidor": "Moderado",
    "objetivo_principal": "Criar reserva de emergência para 6 meses",
    "patrimonio_total": 12000.00,
    "reserva_emergencia_atual": 3500.00
}

transacoes = pd.DataFrame([
    ["2025-03-02", "Projeto design", "Receita", 1500.00],
    ["2025-03-05", "Aluguel", "Moradia", -1200.00],
    ["2025-03-08", "Supermercado", "Alimentação", -420.00],
    ["2025-03-10", "Ifood", "Alimentação", -70.00],
    ["2025-03-12", "Transferência poupança", "Investimento", -300.00],
    ["2025-03-15", "Consultoria", "Receita", 2000.00],
    ["2025-03-18", "Farmácia", "Saúde", -45.00],
    ["2025-03-20", "Uber", "Transporte", -60.00],
    ["2025-03-22", "Conta de luz", "Moradia", -130.00],
    ["2025-03-25", "Ifood", "Alimentação", -55.00],
    ["2025-03-28", "Projeto design", "Receita", 1500.00],
    ["2025-03-30", "Reserva emergência", "Investimento", -400.00]
], columns=["data", "descricao", "categoria", "valor"])

historico = pd.DataFrame([
    ["2025-02-12", "Quanto devo guardar por mês para reserva?", "O ideal é guardar de 10% a 20% da sua renda mensal. No seu caso, com renda variável, tente guardar um valor fixo quando receber."],
    ["2025-02-18", "Onde devo deixar minha reserva de emergência?", "Em um lugar de fácil acesso, como poupança ou CDB com liquidez diária. O importante é não misturar com investimentos de longo prazo."],
    ["2025-02-25", "E se eu precisar da reserva antes?", "Sem problemas! A reserva é feita para ser usada em emergências. O importante é repor depois."]
], columns=["data", "pergunta", "resposta"])

produtos = [
    {"tipo": "Poupança", "descricao": "A mais simples. Rendimento baixo, mas não perde dinheiro. Garantida pelo FGC até R$ 250 mil.", "adequado_para": "Reserva de emergência"},
    {"tipo": "CDB", "descricao": "Título de dívida do banco. Quanto maior o prazo, maior a rentabilidade. Risco baixo (garantido pelo FGC).", "adequado_para": "Curto e médio prazo"},
    {"tipo": "Tesouro Direto (IPCA+)", "descricao": "Título público que rende inflação + juros fixos. Protege contra perda de poder de compra. Indicado para longo prazo.", "adequado_para": "Objetivos de longo prazo (aposentadoria, entrada de imóvel)"},
    {"tipo": "Fundos de investimento", "descricao": "Cesta de ativos gerida por um profissional. Pode ter taxas de administração. Risco varia conforme o tipo de fundo.", "adequado_para": "Quem não quer escolher ativos individualmente"}
]

print("✅ Dados carregados com sucesso")

✅ Dados carregados com sucesso


In [3]:
contexto = f"""
CLIENTE: {perfil['nome']}, {perfil['idade']} anos, perfil {perfil['perfil_investidor']}
OBJETIVO: {perfil['objetivo_principal']}
PATRIMÔNIO: R$ {perfil['patrimonio_total']} | RESERVA: R$ {perfil['reserva_emergencia_atual']}

TRANSAÇÕES RECENTES:
{transacoes.to_string(index=False)}

ATENDIMENTOS ANTERIORES:
{historico.to_string(index=False)}

PRODUTOS DISPONÍVEIS (para fins educativos):
{json.dumps(produtos, indent=2, ensure_ascii=False)}
"""

SYSTEM_PROMPT = f"""Você é Lúcio, um educador financeiro paciente e acolhedor. Siga estas regras rigidamente:

- Use APENAS os dados fornecidos no contexto para responder.
- Não invente informações. Se não encontrar, diga: "Não tenho esse dado, mas posso ajudar com o conceito geral."
- Jamais recomende investimentos. Diga: "Não posso recomendar, mas posso explicar como funciona."
- Responda em no máximo 3 parágrafos, de forma clara e informal.
- Termine perguntando se a pessoa entendeu ou tem mais dúvidas.
- Mantenha o tom acolhedor, como um amigo mais experiente.

CONTEXTO DO CLIENTE:
{contexto}
"""


In [9]:
from getpass import getpass
OPENAI_API_KEY = getpass("🔑 Digite sua chave da OpenAI: ")

def perguntar_lucio(mensagem, historico_chat):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {OPENAI_API_KEY}"
    }
    data = {
        "model": "gpt-3.5-turbo",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": mensagem}
        ],
        "temperature": 0.7,
        "max_tokens": 500
    }
    try:
        response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=data)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]
    except Exception as e:
        return f"Erro: {e}. Verifique sua chave ou créditos."

🔑 Digite sua chave da OpenAI: ··········


In [6]:
def chat_lucio(mensagem, historico):
    resposta = perguntar_lucio(mensagem, historico)
    return resposta

# Interface simples
iface = gr.ChatInterface(
    fn=perguntar_lucio,
    title="🤖 Lúcio – Educador Financeiro",
    description="Pergunte sobre reserva de emergência, orçamento, produtos financeiros. Lúcio ensina, não recomenda investimentos.",
    theme="soft"
)

iface.launch(share=True)  # cria um link público

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8667cfba9bbc4696cc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
